# CourtListener — Keyword / Phrase Search

Use this notebook to search across all opinions already downloaded and indexed
by **courtlistener.ipynb**.

**Prerequisites:** run the pipeline notebook at least once so that
`search_index.db` and the `snapshots/` folder exist.

**How to search:** run the cell below — an input box will appear at the top of
the cell. Type your word or phrase and press **Enter**.

In [53]:
import os, json, sqlite3, textwrap
from dotenv import load_dotenv

# ── Point these at the same folders used by the pipeline notebook ─────────────
DB_PATH       = "search_index.db"
SNAPSHOTS_DIR = "snapshots"

In [54]:
SNIPPET_CHARS = 400
TOP_N         = 20


def _load_snapshot_index(snapshots_dir):
    if not os.path.exists(snapshots_dir):
        return {}
    files = sorted(
        f for f in os.listdir(snapshots_dir)
        if f.startswith("snapshot_") and f.endswith(".json")
    )
    if not files:
        return {}
    with open(os.path.join(snapshots_dir, files[-1]), encoding="utf-8") as f:
        data = json.load(f)
    return {str(c["cluster_id"]): c for c in data}


def search_cases(query, db_path, snapshots_dir, top_n=20, snippet_chars=400):
    if not os.path.exists(db_path):
        print(f"Database not found: {db_path}")
        print("Run courtlistener.ipynb first to download and index cases.")
        return []
    meta = _load_snapshot_index(snapshots_dir)
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT cluster_id, case_name, court, date_filed, text "
        "FROM cases WHERE text LIKE ? LIMIT ?",
        (f"%{query}%", top_n),
    ).fetchall()
    conn.close()
    results = []
    q_lower = query.lower()
    for row in rows:
        text = row["text"] or ""
        pos  = text.lower().find(q_lower)
        if pos == -1:
            continue
        start   = max(0, pos - snippet_chars // 2)
        end     = min(len(text), pos + len(query) + snippet_chars // 2)
        snippet = " ".join(text[start:end].split())
        if start > 0:
            snippet = "…" + snippet
        if end < len(text):
            snippet = snippet + "…"
        cid  = str(row["cluster_id"])
        snap = meta.get(cid, {})
        results.append({
            "case_name":     snap.get("case_name")     or row["case_name"],
            "court":         snap.get("court")         or row["court"],
            "date_filed":    snap.get("date_filed")    or row["date_filed"],
            "docket_number": snap.get("docket_number") or "N/A",
            "citation":      snap.get("citation")      or "N/A",
            "cluster_id":    cid,
            "snippet":       snippet,
        })
    return results


# ── Prompt and run ─────────────────────────────────────────────────────────────
query = input("Search for: ").strip()

if not query:
    print("No query entered.")
else:
    matches = search_cases(query, DB_PATH, SNAPSHOTS_DIR, TOP_N, SNIPPET_CHARS)
    SEP = "─" * 72
    if not matches:
        print(f'No matches found for: "{query}"')
    else:
        print(f'\nFound {len(matches)} case(s) containing "{query}":\n')
        for m in matches:
            print(SEP)
            print(f"  Case:    {m['case_name']}")
            print(f"  Court:   {m['court']}")
            print(f"  Filed:   {m['date_filed']}    Docket: {m['docket_number']}    Citation: {m['citation']}")
            print(f"  ID:      {m['cluster_id']}")
            print(f"  Passage:")
            print(textwrap.fill(m["snippet"], width=68, initial_indent="    ", subsequent_indent="    "))
        print(SEP)


Found 16 case(s) containing "Substantiality":

────────────────────────────────────────────────────────────────────────
  Case:    Medical Imaging & Technology Alliance v. Library of Congress
  Court:   District Court, District of Columbia
  Filed:   2025-07-21    Docket: Civil Action No. 2022-0499    Citation: N/A
  ID:      10637866
  Passage:
    …d character of the use, including whether such use is of a
    commercial nature or is for nonprofit educational purposes; (2)
    the nature of the copyrighted work; (3) the amount and
    substantiality of the portion used in relation to the
    copyrighted work as a whole; and (4) the effect of the use upon
    the potential market for or value of the copyrighted work. Id. §
    107. “Wh…
────────────────────────────────────────────────────────────────────────
  Case:    American Society for Testing and Materials v. Public.Resource.Org, Inc.
  Court:   Court of Appeals for the D.C. Circuit
  Filed:   2023-09-12    Docket: 22-7063    Ci